# Inverse mapping: results

The xyz to flux-coordinate inverse map on the finite-beta equilibrium A at the machine scale (`inverse_mapping/data/desc_equilibrium_a_machine_scale.npz`): its flux surfaces, the residuals of the soak campaign in `stress_tests/`, and the campaign's test points on one cross-section.

In [ ]:
import pathlib

import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.ideal_mhd import (
    desc_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.ideal_mhd import (
    plot as ideal_mhd_plot,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    pubfig,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_inverse_mapping as plots,
)

# Anchor the output directory to the package location, not the kernel's cwd, so the
# notebook runs from any working directory.
ROOT = pathlib.Path(plots.__file__).parents[2]
OUTPUT_DIR = ROOT / "results_plots_tables" / "results" / "inverse_mapping"
assert OUTPUT_DIR.is_dir(), OUTPUT_DIR


def save_pdf(figure, name: str) -> None:
    figure.savefig(OUTPUT_DIR / f"{name}.pdf")


pubfig.apply_style()
EQUILIBRIUM_PATH = (
    ROOT / "inverse_mapping" / "data" / "desc_equilibrium_a_machine_scale.npz"
)
# Written by `stress_tests/run_soak.py`, not by this notebook.
SOAK_CSV = OUTPUT_DIR / "inverse_map_soak_residuals.csv"
equilibrium = plots.load_equilibrium_npz(EQUILIBRIUM_PATH)
coefficients = equilibrium.coefficients
profiles = [
    name
    for name in ("pressure", "iota", "current")
    if getattr(equilibrium, f"{name}_profile") is not None
]
print(
    f"equilibrium: {EQUILIBRIUM_PATH.name}, nfp = {equilibrium.n_field_periods}, "
    f"{coefficients.r_lmn.shape[0]} R modes, "
    f"Psi = {float(equilibrium.total_toroidal_flux):.4g} Wb, profiles: {profiles}"
)

## 1. Flux surfaces

Six toroidal sections over one field period, in the style of `desc.plotting.plot_surfaces`: constant-`rho` surfaces (boundary emphasized), constant-`theta` contours and the magnetic axis.

In [ ]:
figure, _ = ideal_mhd_plot.plot_surfaces(equilibrium)
save_pdf(figure, "inverse_map_flux_surfaces")

## 2. Soak-campaign residuals

`stress_tests/run_soak.py`, run separately, writes `inverse_map_soak_residuals.csv`. It inverts test points of known inside/outside status at `tol / 100` (on measurement chords, near the axis, uniformly inside, and straddling the boundary) and keeps per shard its failing points or, if none, its five worst-converged ones. The table is therefore the campaign's extreme tail, not a sample.

Markers are coloured by generator, failures are ringed in red, and the dashed line is the acceptance tolerance. The whole tail sits decades under it, with no failures.

In [ ]:
table = pd.read_csv(SOAK_CSV).sort_values(
    "final_residual", ascending=False, ignore_index=True
)
summary = {
    "campaign": str(table["campaign"].iloc[0]),
    "equilibrium": str(table["equilibrium"].iloc[0]),
    "n_shards": int(table["n_shards"].iloc[0]),
    "n_shards_recorded": int(table["n_shards_recorded"].iloc[0]),
    "n_points_total": int(
        table["n_batches"].iloc[0] * table["n_points_per_batch"].iloc[0]
    ),
    "n_recorded": int(len(table)),
    "n_failures": int((table["failure_kind"] != "none").sum()),
    "tol": float(table["tol"].iloc[0]),
    "worst_residual": float(table["final_residual"].max()),
}
print(
    f"campaign {summary['campaign']} on {summary['equilibrium']}: "
    f"{summary['n_points_total']:.3g} points over "
    f"{summary['n_shards_recorded']}/{summary['n_shards']} shards, "
    f"{summary['n_failures']} failures in {summary['n_recorded']} recorded points"
)
print(
    f"worst residual {summary['worst_residual']:.3e} m against tol "
    f"{summary['tol']:g} m; worst per generator:"
)
print(table.groupby("generator")["final_residual"].max().to_string())
assert summary["n_shards_recorded"] == summary["n_shards"], (
    "the campaign in the CSV is incomplete; rerun or point SOAK_CSV at a finished one"
)
assert summary["worst_residual"] < summary["tol"], (
    "every recorded point must sit below the acceptance tolerance"
)
table.head(10)

In [ ]:
figure, (panel,) = pubfig.subplots(1, 1, width="single", aspect=0.85)
panel.set_yscale("log")
for index, (generator, group) in enumerate(table.groupby("generator", sort=True)):
    panel.plot(
        group["final_rho"],
        group["final_residual"],
        "o",
        ms=3.0,
        lw=0.0,
        color=pubfig.CATEGORICAL_PALETTE[index % len(pubfig.CATEGORICAL_PALETTE)],
        label=str(generator).replace("_", " "),
    )
failed = table[table["failure_kind"] != "none"]
if len(failed):
    panel.plot(
        failed["final_rho"],
        failed["final_residual"],
        "o",
        ms=6.0,
        mfc="none",
        mec="red",
        mew=1.0,
        label="failure",
    )
tolerance = summary["tol"]
panel.axhline(
    tolerance, color="0.25", lw=1.2, ls=(0, (4, 2)), label=f"tolerance {tolerance:g} m"
)
# The boundary is where the inside/outside verdict is decided, so it is worth
# seeing which side the worst points sit on.
panel.axvline(1.0, color="0.7", lw=0.8, zorder=0)
panel.set_ylim(top=10.0 * tolerance)
panel.set_xlabel(r"$\rho$ of the converged iterate")
panel.set_ylabel("round-trip residual (m)")
figure.legend(
    handles=panel.get_legend_handles_labels()[0], ncol=2, loc="outside lower center"
)
save_pdf(figure, "inverse_map_soak_residuals")

## 3. Test points of the campaign

The campaign's batches regenerated from their seeds (`soak_utils.sample_inverse_map_test_points`, same mix as the run), keeping the points within `ZETA_HALF_WIDTH` of four `phi` planes over half a field period, which by stellarator symmetry holds every distinct cross-section, and drawing them over those sections, one colour per generator (the colours of the residual figure). Chords appear as dotted lines, the near-axis points as a blob around the axis, the boundary-straddling points on the last closed flux surface.

In [ ]:
import operator

import numpy as np
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.results.inverse_mapping.stress_tests import (  # noqa: E501
    run_soak,
    soak_utils,
)

# A point belongs to a section when its toroidal angle is within ZETA_HALF_WIDTH of
# it; the section barely changes over that angle, so the points sit on the drawn
# surfaces.
ZETA_HALF_WIDTH = 0.01
sections = np.linspace(0.0, np.pi / equilibrium.n_field_periods, 4)
seeds = np.random.default_rng(run_soak.SEED_OF_SEEDS).integers(
    0, 2**63, int(table["n_batches"].iloc[0])
)
points_rz, zeta, generator_type = [], [], []
for seed in seeds:
    batch = soak_utils.sample_inverse_map_test_points(
        coefficients,
        np.random.default_rng(int(seed)),
        int(table["n_points_per_batch"].iloc[0]),
        boundary_offset=summary["tol"],
    )
    xyz = batch.points_xyz
    batch_zeta = np.mod(np.arctan2(xyz[:, 1], xyz[:, 0]), 2.0 * np.pi)
    angular_distance = np.abs(
        (batch_zeta[:, None] - sections + np.pi) % (2.0 * np.pi) - np.pi
    )
    near_section = (angular_distance < ZETA_HALF_WIDTH).any(axis=1)
    points_rz.append(
        np.column_stack(
            [np.hypot(xyz[near_section, 0], xyz[near_section, 1]), xyz[near_section, 2]]
        )
    )
    zeta.append(batch_zeta[near_section])
    generator_type.append(batch.generator_type[near_section])
points_rz = np.concatenate(points_rz)
zeta = np.concatenate(zeta)
generator_type = np.concatenate(generator_type)

# One row at a common scale: panel widths follow each section's R extent and the Z
# axis is shared, so equal aspect gives every section the same metres per inch.
section_widths = []
boundary_theta = np.linspace(0.0, 2.0 * np.pi, 721)
for section in sections:
    boundary_r, _ = desc_utils.evaluate_rz(
        coefficients,
        np.ones_like(boundary_theta),
        boundary_theta,
        np.full_like(boundary_theta, section),
    )
    section_widths.append(float(np.ptp(np.asarray(boundary_r))))
figure, panels = pubfig.subplots(
    1,
    len(sections),
    width="double",
    aspect=2.9,
    sharey=True,
    gridspec_kw={"width_ratios": section_widths},
)
ideal_mhd_plot.plot_surfaces(
    equilibrium,
    rho=5,
    theta=0,
    phi=sections,
    ax=panels,
    rho_color="0.75",
    rho_lw=0.8,
    lcfs_color="0.25",
    lcfs_lw=1.0,
    axis_color="0.25",
    axis_size=10,
)
# Alphabetical by label, the order of the residual figure, so colours match.
generators = sorted(soak_utils.GENERATOR_LABELS.items(), key=operator.itemgetter(1))
for panel_index, (panel, section) in enumerate(zip(panels, sections)):
    panel.tick_params(axis="x", labelrotation=90)
    panel.set_xlabel("")
    if panel_index > 0:
        panel.set_ylabel("")
    in_section = (
        np.abs((zeta - section + np.pi) % (2.0 * np.pi) - np.pi) < ZETA_HALF_WIDTH
    )
    for index, (generator, label) in enumerate(generators):
        mask = in_section & (generator_type == generator)
        panel.plot(
            points_rz[mask, 0],
            points_rz[mask, 1],
            "o",
            ms=1.2,
            lw=0.0,
            mew=0.0,
            color=pubfig.CATEGORICAL_PALETTE[index],
            label=label.replace("_", " ") if panel_index == 0 else None,
        )
    counts = {
        label: int((in_section & (generator_type == generator)).sum())
        for generator, label in generators
    }
    print(f"phi = {section / np.pi:.2f} pi: {counts}")
figure.supxlabel(r"$R$ (m)")
figure.legend(
    handles=panels[0].get_legend_handles_labels()[0],
    ncol=4,
    loc="outside upper center",
    markerscale=6.0,
)
save_pdf(figure, "inverse_map_test_points")

## 4. Cost against DESC's `map_coordinates`

The same inversion done two ways on ten interior points of this equilibrium: DESC's `Equilibrium.map_coordinates`, which needs a live `Equilibrium` object, timed with and without the object's construction, and the package's `xyz_to_flux_coordinates` on the bare coefficient arrays, eager and under `jax.jit`.

Kept equal so the times compare: the equilibrium (the modes DESC builds are asserted to be the package's), the points, the tolerance and the iteration budget, and the accuracy, judged for every variant by the same round trip through `evaluate_rz`. Every call is blocked until its result is ready. The first call of each variant carries its own trace and compile cost and is reported apart from the median of the repeats; `map_coordinates` defines its jitted functions inside the call, so it pays that cost every time, whereas the package map pays it once per process. DESC's optional backtracking line search is included as the like-for-like counterpart of the package's damped step.

In [ ]:
import functools
import statistics
import time

import desc.equilibrium
import jax
from plasma_state_estimation_for_diagnostic_placement_optimization.ideal_mhd import (
    desc_types,
)

TIMING_N_POINTS = 10
TIMING_TOL = 1e-12
TIMING_MAXITER = 50
TIMING_N_REPEATS = 5
TIMING_CSV = OUTPUT_DIR / "inverse_map_timing_vs_desc.csv"

# The DESC constructor takes the resolution, not the mode tables; read it off the
# tables and check below that DESC lays its modes out in the same order.
r_modes = np.asarray(coefficients.r_modes)
resolution = desc_types.DescResolution(
    radial_resolution=int(r_modes[:, 0].max()),
    poloidal_resolution=int(np.abs(r_modes[:, 1]).max()),
    toroidal_resolution=int(np.abs(r_modes[:, 2]).max()),
    n_field_periods=coefficients.n_field_periods,
    is_stellarator_symmetric=desc_utils.is_stellarator_symmetric(coefficients),
)

# Interior points from the forward map, so both inversions start from a point whose
# flux coordinates are known and the round trip is judged on the same forward map.
timing_rng = np.random.default_rng(0)
timing_rho = timing_rng.uniform(0.2, 0.85, TIMING_N_POINTS)
timing_theta = timing_rng.uniform(0.0, 2.0 * np.pi, TIMING_N_POINTS)
timing_zeta = timing_rng.uniform(0.0, 2.0 * np.pi, TIMING_N_POINTS)
timing_r, timing_z = desc_utils.evaluate_rz(
    coefficients, timing_rho, timing_theta, timing_zeta
)
timing_r, timing_z = np.asarray(timing_r), np.asarray(timing_z)
timing_points_xyz = np.column_stack(
    [timing_r * np.cos(timing_zeta), timing_r * np.sin(timing_zeta), timing_z]
)
timing_coords_rpz = np.column_stack([timing_r, timing_zeta, timing_z])


def timing_round_trip(rho_out, theta_out, zeta_out) -> float:
    """Largest real-space distance between a point and its inverted-then-mapped image."""
    r_back, z_back = desc_utils.evaluate_rz(
        coefficients, np.asarray(rho_out), np.asarray(theta_out), np.asarray(zeta_out)
    )
    return float(
        np.max(np.hypot(np.asarray(r_back) - timing_r, np.asarray(z_back) - timing_z))
    )


def timed(function):
    """Wall-clock of the first call and the median of further calls, in seconds."""
    start = time.perf_counter()
    result = function()
    first = time.perf_counter() - start
    laps = []
    for _ in range(TIMING_N_REPEATS):
        start = time.perf_counter()
        result = function()
        laps.append(time.perf_counter() - start)
    return first, statistics.median(laps), result


def build_desc_equilibrium(check_orientation: bool) -> desc.equilibrium.Equilibrium:
    """The same geometry and profiles as a live DESC object, as DESC's constructor
    takes them; `check_orientation` is DESC's default True."""
    return desc.equilibrium.Equilibrium(
        R_lmn=np.asarray(coefficients.r_lmn),
        Z_lmn=np.asarray(coefficients.z_lmn),
        L_lmn=np.asarray(coefficients.lambda_lmn),
        L=resolution.radial_resolution,
        M=resolution.poloidal_resolution,
        N=resolution.toroidal_resolution,
        NFP=resolution.n_field_periods,
        sym=resolution.is_stellarator_symmetric,
        spectral_indexing=resolution.spectral_indexing,
        Psi=float(equilibrium.total_toroidal_flux),
        pressure=desc_utils.desc_spline_profile_to_desc_object(
            equilibrium.pressure_profile
        ),
        current=(
            desc_utils.desc_spline_profile_to_desc_object(equilibrium.current_profile)
            if equilibrium.current_profile is not None
            else None
        ),
        iota=(
            desc_utils.desc_spline_profile_to_desc_object(equilibrium.iota_profile)
            if equilibrium.iota_profile is not None
            else None
        ),
        check_orientation=check_orientation,
    )


def desc_map_coordinates(built: desc.equilibrium.Equilibrium, **kwargs):
    out, (_, n_iterations) = built.map_coordinates(
        timing_coords_rpz,
        inbasis=("R", "phi", "Z"),
        outbasis=("rho", "theta", "zeta"),
        period=(np.inf, 2.0 * np.pi, np.inf),
        tol=TIMING_TOL,
        maxiter=TIMING_MAXITER,
        full_output=True,
        **kwargs,
    )
    return np.asarray(jax.block_until_ready(out)), int(np.max(np.asarray(n_iterations)))


def package_eager():
    return jax.block_until_ready(
        desc_utils.xyz_to_flux_coordinates(
            coefficients, timing_points_xyz, tol=TIMING_TOL, maxiter=TIMING_MAXITER
        )
    )


package_jitted = soak_utils.make_jitted_inverse_map(
    coefficients, tol=TIMING_TOL, maxiter=TIMING_MAXITER
)
package_jitted_with_guess = jax.jit(
    functools.partial(
        desc_utils.xyz_to_flux_coordinates,
        coefficients,
        tol=TIMING_TOL,
        maxiter=TIMING_MAXITER,
    )
)
timing_guess = np.stack(
    desc_utils.initial_guess(coefficients, timing_points_xyz), axis=1
)


def package_jit():
    return jax.block_until_ready(package_jitted(timing_points_xyz))


def package_jit_with_guess():
    return jax.block_until_ready(
        package_jitted_with_guess(timing_points_xyz, guess=timing_guess)
    )


timing_rows = []
for check_orientation in (True, False):
    first, repeated, built = timed(
        functools.partial(build_desc_equilibrium, check_orientation)
    )
    assert np.array_equal(np.asarray(built.R_basis.modes), r_modes)
    assert np.array_equal(
        np.asarray(built.Z_basis.modes), np.asarray(coefficients.z_modes)
    )
    timing_rows.append(
        {
            "variant": f"DESC Equilibrium construction (check_orientation={check_orientation})",
            "first_call_s": first,
            "repeated_call_s": repeated,
            "max_round_trip_m": np.nan,
            "max_iterations": np.nan,
        }
    )
desc_equilibrium = build_desc_equilibrium(True)

# DESC's optional backtracking line search halves the step, as the package's damping
# does, so it is the like-for-like setting; DESC's default has none.
for label, kwargs in (
    ("DESC map_coordinates (default, no line search)", {}),
    ("DESC map_coordinates (maxiter_ls=4, alpha=0.5)", {"maxiter_ls": 4, "alpha": 0.5}),
):
    first, repeated, (out, n_iterations) = timed(
        functools.partial(desc_map_coordinates, desc_equilibrium, **kwargs)
    )
    timing_rows.append(
        {
            "variant": label,
            "first_call_s": first,
            "repeated_call_s": repeated,
            "max_round_trip_m": timing_round_trip(out[:, 0], out[:, 1], out[:, 2]),
            "max_iterations": n_iterations,
        }
    )

for label, function in (
    ("package xyz_to_flux_coordinates (eager)", package_eager),
    ("package xyz_to_flux_coordinates (jit)", package_jit),
    (
        "package xyz_to_flux_coordinates (jit, precomputed guess)",
        package_jit_with_guess,
    ),
):
    first, repeated, flux = timed(function)
    timing_rows.append(
        {
            "variant": label,
            "first_call_s": first,
            "repeated_call_s": repeated,
            "max_round_trip_m": timing_round_trip(flux.rho, flux.theta, flux.zeta),
            "max_iterations": np.nan,
        }
    )

timing = pd.DataFrame(timing_rows)
timing.to_csv(TIMING_CSV, index=False)
assert (timing["max_round_trip_m"].dropna() < TIMING_TOL).all(), (
    "every variant must reach the requested tolerance for the times to compare"
)
print(
    f"{TIMING_N_POINTS} points, tol {TIMING_TOL:g} m, maxiter {TIMING_MAXITER}, "
    f"median of {TIMING_N_REPEATS} repeats after the first call"
)
timing